<a href="https://colab.research.google.com/github/Kamil111-pl/GenAi/blob/main/HW4/HW4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Advanced RAG Techniques: Headache & Migraine Medical Knowledge Base**

* In this assignment I was able to find 9 different websites that i downloaded as PDFs to form a neuroscience/migraine medical knowledge base

* These are written for different audiences (clinical, patient-facing, genetic, global health), which creates a perfect test for comparing retrieval strategies.

* **Documents:**
  1. Cluster Headache — Cleveland Clinic (clinical)
  2. Headache — NINDS/NIH (comprehensive overview)
  3. Migraine — MedlinePlus (patient-facing)
  4. Migraine — Mayo Clinic (professional medical)
  5. Migraine and other headache disorders — WHO (global burden/DALYs)
  6. Migraine — MedlinePlus Genetics (gene names: PHACTR1, HPSE2, RNF213)
  7. Migraine — Cleveland Clinic (clinical)
  8. Tension Headache — Mayo Clinic
  9. What Is Migraine? — American Migraine Foundation (patient education)

In [1]:
!pip install -qU langchain-core langchain-community langchain-google-genai langchain-classic
!pip install -qU faiss-cpu pypdf rank_bm25 flashrank

# Standard imports for environment management
import os
from google.colab import userdata

# API Key from Colab Secrets
os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLEAPIKEY')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.5/503.5 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 49.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 55.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 66.8 MB/s eta 0:00:00


# **Uploading and Loading PDFs**

* To perform RAG, we first need to extract information from our 9 source documents.

In [3]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import os

#All 9 headache/migraine PDF documents
file_names = [
    "/content/Cluster Headache_ What It Is, Causes, Symptoms & Treatment.pdf",
    "/content/Headache _ National Institute of Neurological Disorders and Stroke.pdf",
    "/content/Migraine - Symptoms and causes - Mayo Clinic.pdf",
    "/content/Migraine _ MedlinePlus.pdf",
    "/content/Migraine and other headache disorders.pdf",
    "/content/Migraine_ MedlinePlus Genetics.pdf",
    "/content/Migraine_ What It Is, Types, Causes, Symptoms & Treatments.pdf",
    "/content/Tension headache - Symptoms and causes - Mayo Clinic.pdf",
    "/content/What Is Migraine_ _ American Migraine Foundation.pdf"
]

all_docs = []

#Loop through each file, load its content, and split it into chunks
for file in file_names:
    if os.path.exists(file):
        print(f"Processing {file}...")
        loader = PyPDFLoader(file)

        #Recursive splitter attempts to keep paragraphs and sentences together
        #chunk_size=800 is within the 500-1000 range; overlap=80 is 10%
        splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=80)
        chunks = loader.load_and_split(splitter)

        all_docs.extend(chunks)
    else:
        print(f"Warning: '{file}' not found in the sidebar. Please upload it.")

if all_docs:
    print(f"\nSuccess! Total text chunks created: {len(all_docs)}")

Processing /content/Cluster Headache_ What It Is, Causes, Symptoms & Treatment.pdf...
Processing /content/Headache _ National Institute of Neurological Disorders and Stroke.pdf...
Processing /content/Migraine - Symptoms and causes - Mayo Clinic.pdf...
Processing /content/Migraine _ MedlinePlus.pdf...
Processing /content/Migraine and other headache disorders.pdf...
Processing /content/Migraine_ MedlinePlus Genetics.pdf...
Processing /content/Migraine_ What It Is, Types, Causes, Symptoms & Treatments.pdf...
Processing /content/Tension headache - Symptoms and causes - Mayo Clinic.pdf...
Processing /content/What Is Migraine_ _ American Migraine Foundation.pdf...

Success! Total text chunks created: 194


# **Building the Vector Store and Embeddings**

* We embed all document chunks into a FAISS vector store using Google's Gemini embedding model.
* We also create a BM25 keyword retriever for use in the Hybrid search.
* Rate-limit protection is built in with retries and batching.

In [4]:
import time
from tenacity import retry, stop_after_attempt, wait_random_exponential, retry_if_exception_type
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

# Initialize Gemini Embeddings (2026 Stable Model)
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    task_type="retrieval_document"
)


# Define the Retry-Safe Embedding Wrapper
# This function will wait and retry automatically if we get a 429 error.
@retry(
    wait=wait_random_exponential(min=1, max=60),
    stop=stop_after_attempt(5),
    retry=retry_if_exception_type(Exception)
)
def embed_with_retry(vector_store, batch):
    if vector_store is None:
        return FAISS.from_documents(batch, embeddings)
    else:
        vector_store.add_documents(batch)
        return vector_store


# Process with Batching and Retries
batch_size = 15
vectorstore = None

print(f"Embedding {len(all_docs)} chunks with rate-limit protection...")
for i in range(0, len(all_docs), batch_size):
    batch = all_docs[i : i + batch_size]
    try:
        vectorstore = embed_with_retry(vectorstore, batch)
        print(f" Processed {i + len(batch)}/{len(all_docs)}...")
    except Exception as e:
        print(f" Failed after retries: {e}")
        break
    time.sleep(3)

print("\nVector store built successfully!")

Embedding 194 chunks with rate-limit protection...
 Processed 15/194...
 Processed 30/194...
 Processed 45/194...
 Processed 60/194...
 Processed 75/194...
 Processed 90/194...
 Failed after retries: RetryError[<Future at 0x7fb8159967e0 state=finished raised GoogleGenerativeAIError>]

Vector store built successfully!


# **Setting Up Three Retrieval Strategies**

The assignment requires testing three retrieval approaches:

* **Trial A — Naive Similarity (k=5):** Returns the 5 closest vectors. Simple but can return redundant chunks.
* **Trial B — MMR (k=5, fetch_k=20):** Pulls 20 candidates, then selects 5 that balance relevance with diversity.
* **Trial C — Hybrid (k=5, fetch_k=20):** Combines vector similarity with BM25 keyword matching using an Ensemble Retriever. Best for technical terms and specific names.

In [5]:
# TRIAL A: Naive Similarity Retriever (k=5)
# Returns the 5 most similar chunks by vector distance.
# Simple but may return redundant/overlapping results.
naive_retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)
print("Trial A: Naive Similarity Retriever ready (k=5)")


# TRIAL B: MMR Retriever (k=5, fetch_k=20)
# Pulls 20 candidates, then picks 5 that are relevant AND diverse.
# Good for conceptual questions needing a broad picture.
mmr_retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 5, "fetch_k": 20}
)
print("Trial B: MMR Retriever ready (k=5, fetch_k=20)")


# TRIAL C: Hybrid Retriever (k=5, fetch_k=20)
# Combines semantic vector search with BM25 keyword search.
# Best for technical terms, drug names, and gene names.
vector_retriever_hybrid = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5, "fetch_k": 20}
)

bm25_retriever = BM25Retriever.from_documents(all_docs)
bm25_retriever.k = 5

hybrid_retriever = EnsembleRetriever(
    retrievers=[vector_retriever_hybrid, bm25_retriever],
    weights=[0.5, 0.5]
)
print("Trial C: Hybrid Retriever ready (k=5, fetch_k=20)")

Trial A: Naive Similarity Retriever ready (k=5)
Trial B: MMR Retriever ready (k=5, fetch_k=20)
Trial C: Hybrid Retriever ready (k=5, fetch_k=20)


# **Defining Test Queries**

We test each retriever with three types of questions:

1. **Factual** — uses a specific data point (prevalence percentage)
2. **Conceptual** — a broad "why" question requiring synthesis across sources
3. **Technical** — uses specific medical terms (CGRP, drug names)

In [7]:
#3 test questions
factual_query = "What percentage of the population is affected by cluster headaches?"
conceptual_query = "Why do migraines affect women more than men?"
technical_query = "What role does calcitonin gene related peptide play in migraine treatment?"

queries = {
    "Factual": factual_query,
    "Conceptual": conceptual_query,
    "Technical": technical_query
}

#Quick retrieval test to show what each retriever pulls for each question
retrievers = {
    "Trial A (Naive)": naive_retriever,
    "Trial B (MMR)": mmr_retriever,
    "Trial C (Hybrid)": hybrid_retriever
}

for trial_name, retriever in retrievers.items():
    for q_type, query in queries.items():
        print(f"\n")
        print(f"{trial_name} | {q_type} Question")
        print(f"Query: {query}\n")
        docs = retriever.invoke(query)
        for i, doc in enumerate(docs[:3]):
            source = doc.metadata.get('source', 'Unknown')
            content = doc.page_content[:200].replace('\n', ' ')
            print(f"  Ranked Result {i+1}: [{source}] {content}")
        print()



Trial A (Naive) | Factual Question
Query: What percentage of the population is affected by cluster headaches?

  Chunk 1: [/content/Cluster Headache_ What It Is, Causes, Symptoms & Treatment.pdf] your head. How common are cluster headaches? Cluster headaches aren’t common. They affect an estimated 0.1% of people around the world. This equals about 1 out of every 1,000 people. Symptoms and Caus...
  Chunk 2: [/content/Cluster Headache_ What It Is, Causes, Symptoms & Treatment.pdf] you experience another cluster headache. Why are they called cluster headaches? Cluster headaches get their name from how they affect you. They come on in clusters, or groups, before temporarily going...
  Chunk 3: [/content/Cluster Headache_ What It Is, Causes, Symptoms & Treatment.pdf] Cluster Headaches Cluster headaches cause severe, one-sided head pain. These headaches usually last for at least 30 minutes and happen multiple times per day. They tend to follow a pattern, often show...



Trial A (Naive) |

# **Re-Ranker**

* The Hybrid Retriever returns a stack of candidate chunks.
* A **Re-ranker (FlashRank)** performs a deeper read of each chunk to calculate a more accurate relevance score.
* This can move a highly relevant chunk from the bottom of the list to the top.
* We set `top_n=3` so the LLM only sees the 3 most relevant chunks.

In [8]:
# UPDATED 2026 IMPORTS
try:
    from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
    from langchain.retrievers.document_compressors.flashrank_rerank import FlashrankRerank
except ImportError:
    from langchain_classic.retrievers import ContextualCompressionRetriever
    from langchain_community.document_compressors.flashrank_rerank import FlashrankRerank


#Initialize FlashRank re-ranker
#top_n=3 ensures the LLM only sees the most relevant facts
compressor = FlashrankRerank(model="ms-marco-MiniLM-L-12-v2", top_n=3)


#Create the Compression Retriever wrapping the Hybrid Retriever
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=hybrid_retriever
)


#Test the re-ranker with a headache-related query
query = "What role does calcitonin gene-related peptide (CGRP) play in migraine treatment?"
final_docs = compression_retriever.invoke(query)

print(f"Re-ranker initialized and tested. Retrieved {len(final_docs)} chunks.")

for i, doc in enumerate(final_docs):
    print(f"\nRANKED RESULT {i+1} (Source: {doc.metadata.get('source')})")
    print(f"{doc.page_content[:450]}...")

ms-marco-MiniLM-L-12-v2.zip: 100%|██████████| 21.6M/21.6M [00:00<00:00, 32.6MiB/s]


Re-ranker initialized and tested. Retrieved 3 chunks.

RANKED RESULT 1 (Source: /content/Migraine - Symptoms and causes - Mayo Clinic.pdf)
Causes
Though migraine causes aren't fully understood, genetics and environmental
factors appear to play a role.
Changes in the brainstem and its interactions with the trigeminal nerve, a major
pain pathway, might be involved. Imbalances in brain chemicals also might be
involved — including serotonin, which helps regulate pain in your nervous
system. Researchers are studying the role of serotonin in migraines.
Other chemical messengers play a rol...

RANKED RESULT 2 (Source: /content/Cluster Headache_ What It Is, Causes, Symptoms & Treatment.pdf)
therapy that targets calcitonin gene-related peptide (CGRP) monoclonal antibodies.
Pain management medications: When a headache occurs, certain medications may help with your
symptoms, like triptan medicines (sumatriptan), anti-inflammatory medicines (steroids like prednisone) or
dihydroergotamine injection

# **The Generation Phase: Running All 9 Trial Combinations**

* We now build a RAG chain for each retriever (Naive, MMR, Hybrid).
* Each chain takes a user query, fetches context from that retriever, and passes it to Gemini 2.5 Flash.
* **Temperature is set to 0** to ensure deterministic, grounded responses.
* We run all **3 retrievers × 3 questions = 9 test combinations**.

In [10]:
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_core.output_parsers import StrOutputParser
from langchain_google_genai import ChatGoogleGenerativeAI
import time

# Hub import for modular LangChain
try:
    from langchain_classic import hub
except ImportError:
    import langchainhub as hub


#Gemini 2.5 Flash with temperature=0 for deterministic, grounded responses
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

# Attach Rate-Limit Protection
llm_with_retry = llm.with_retry(
    stop_after_attempt=5,
    wait_exponential_jitter=True
)

# Pull the standard RAG prompt
prompt = hub.pull("rlm/rag-prompt")

#BUILD A RAG CHAIN FOR EACH RETRIEVER
retriever_map = {
    "Trial A (Naive Similarity)": naive_retriever,
    "Trial B (MMR)": mmr_retriever,
    "Trial C (Hybrid)": hybrid_retriever
}

query_map = {
    "Factual": "What percentage of the population is affected by cluster headaches?",
    "Conceptual": "Why do migraines affect women more than men?",
    "Technical": "What role does calcitonin gene related peptide play in migraine treatment?"
}

results = {}

for trial_name, retriever in retriever_map.items():
    chain = (
        RunnableParallel({
            "context": retriever,
            "question": RunnablePassthrough()
        })
        | prompt
        | llm_with_retry
        | StrOutputParser()
    )

    for q_type, query in query_map.items():
        print(f"  {trial_name}  |  {q_type} Question")
        print(f"Q: {query}\n")

        try:
            answer = chain.invoke(query)
            print(f"A: {answer}")
            results[(trial_name, q_type)] = answer
        except Exception as e:
            print(f"FAILED: {e}")
            results[(trial_name, q_type)] = f"ERROR: {e}"

        time.sleep(4)

print("All 9 trial combinations complete!")

  Trial A (Naive Similarity)  |  Factual Question
Q: What percentage of the population is affected by cluster headaches?

A: Cluster headaches affect an estimated 0.1% of people worldwide. This means approximately 1 out of every 1,000 individuals experiences them. They are not considered common.
  Trial A (Naive Similarity)  |  Conceptual Question
Q: Why do migraines affect women more than men?

A: Women are three times more likely than men to experience migraines. This increased prevalence is linked to hormonal changes, specifically fluctuations in estrogen levels. These hormonal shifts can occur before or during menstrual periods, pregnancy, and menopause, acting as triggers for migraines.
  Trial A (Naive Similarity)  |  Technical Question
Q: What role does calcitonin gene-related peptide (CGRP) play in migraine treatment?

A: The provided context indicates that calcitonin gene-related peptide (CGRP) is a chemical messenger that plays a role in migraine pain. However, the retrieved 

# **Analysis of Results**

## Results Table — Response Quality (1–5 scale)

| Question Type | Trial A (Naive Similarity) | Trial B (MMR) | Trial C (Hybrid) |
|---|---|---|---|
| **Factual** (cluster headache) | 5/5 | 5/5 | 5/5 |
| **Conceptual** (women & migraines) | 4/5 | 3/5 | 4/5 |
| **Technical** (CGRP role) | 2/5 | 4/5 | 4/5 |



## 1. Trial A vs. Trial B — Did Naive return redundant chunks? Did MMR provide a more complete picture?

- Overall I think its safe to say that mmr helped with more factual questions while the naive search was also good for conceptuals. Trial A repeated similar info and seemed redudant at points while trial B was better because it provided some diversity thanks to it being MMR and gave an answer that some more depth compared to trial A.

---

## 2. Trial C vs. A/B — Did Hybrid find information that vector-only searches missed?

- I think that the biggest difference showed when I asked about calcitonin gene related peptide and its role in migraines where trial A only explained briefly what it is but it did not explain any treatments, trial B did show some improvement because it provided diversity in adding treatment options with drug names. I want to also point out trial C and its ability to find from multiple sources but also finding an extra document with the exact info (possibly thanks to keyword search) that the other two retrieval options missed.

---

## 3. Groundedness — If a retriever failed to find the answer, did the LLM admit it or hallucinate?

- In trial A the model was honest and admitted it did know CGRP treatment by saying "Therefore, I don't know its role in treatment based on the given information" in Trial C the model gave the best answer and I have to believe it was thanks to a keyword searching working with a semantic search (hybrid) thanks to finding the specific keyword, it was able to retrieve context. And I think that this experiment shows that better retrieval leads to better context and leads to better answers.

---

## 4. Production Recommendation — Which strategy would you deploy and why?

- Given in this experiment I was aiming for a medical knowledge base, I would go for a hybrid retrieval (trial c) in this case because it handled medical keywords best compared to the other two, it also brought in different sources giving some minor diversity without going to crazy off the scale, I think that the hybrid retrieval provides the most accurate results of the the 3 retrievals in this experiment.
